In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import (                           
    StructType, StructField, StringType, IntegerType,
    DoubleType, DateType
)
from pyspark.sql.functions import (                     
    col, when, concat_ws, initcap, upper, lower, trim,
    year, month, datediff, current_date, round as spark_round,
    count, sum as spark_sum, avg, min as spark_min, max as spark_max,
    count_distinct, collect_set, row_number, rank, dense_rank,
    lag, lead, udf
)
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Gold_Layer_Sales_Analytics") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()


spark.sparkContext.setLogLevel("WARN")

print("SparkSession created:", spark.version)

SparkSession created: 4.2.0


In [39]:
customers_schema = StructType([
    StructField("customer_key",    IntegerType(), True),
    StructField("customer_id",     IntegerType(), True),
    StructField("customer_number", StringType(),  True),
    StructField("first_name",      StringType(),  True),
    StructField("last_name",       StringType(),  True),
    StructField("country",         StringType(),  True),
    StructField("marital_status",  StringType(),  True),
    StructField("gender",          StringType(),  True),
    StructField("birthdate",       DateType(),    True),
    StructField("create_date",     DateType(),    True),
])

In [40]:
products_schema = StructType([
    StructField("product_key",    IntegerType(), True),
    StructField("product_id",     IntegerType(), True),
    StructField("product_number", StringType(),  True),
    StructField("product_name",   StringType(),  True),
    StructField("category_id",    StringType(),  True),
    StructField("category",       StringType(),  True),
    StructField("subcategory",    StringType(),  True),
    StructField("maintenance",    StringType(),  True),
    StructField("cost",           DoubleType(),  True),
    StructField("product_line",   StringType(),  True),
    StructField("start_date",     DateType(),    True),
])

In [41]:
sales_schema = StructType([
    StructField("order_number",   StringType(),  True),
    StructField("product_key",    IntegerType(), True),
    StructField("customer_key",   IntegerType(), True),
    StructField("order_date",     DateType(),    True),
    StructField("shipping_date",  DateType(),    True),
    StructField("due_date",       DateType(),    True),
    StructField("sales_amount",   DoubleType(),  True),
    StructField("quantity",       IntegerType(), True),
    StructField("price",          DoubleType(),  True),
])

In [42]:
df_customers_raw = spark.read.format("csv") \
    .option("header", True) \
    .schema(customers_schema) \
    .load("gold.dim_customers.csv")

In [43]:
df_products_raw = spark.read.format("csv") \
    .option("header", True) \
    .schema(products_schema) \
    .load("gold.dim_products.csv")

In [44]:
df_sales_raw = spark.read.format("csv") \
    .option("header", True) \
    .schema(sales_schema) \
    .load("gold.fact_sales.csv")

In [45]:
print("customers:", df_customers_raw.count())

customers: 18484


In [46]:
print("products :", df_products_raw.count())

products : 295


In [47]:
print("sales:", df_sales_raw.count())

sales: 60398


In [48]:
df_customers_raw.printSchema()

root
 |-- customer_key: integer (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_number: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- country: string (nullable = true)
 |-- marital_status: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- create_date: date (nullable = true)



In [49]:
df_products_raw.printSchema()

root
 |-- product_key: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- product_number: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- subcategory: string (nullable = true)
 |-- maintenance: string (nullable = true)
 |-- cost: double (nullable = true)
 |-- product_line: string (nullable = true)
 |-- start_date: date (nullable = true)



In [50]:
df_sales_raw.printSchema()

root
 |-- order_number: string (nullable = true)
 |-- product_key: integer (nullable = true)
 |-- customer_key: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- shipping_date: date (nullable = true)
 |-- due_date: date (nullable = true)
 |-- sales_amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)



In [51]:
df_customers_raw.show(5, truncate=False)

+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+
|customer_key|customer_id|customer_number|first_name|last_name|country  |marital_status|gender|birthdate |create_date|
+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+
|1           |11000      |AW00011000     |Jon       |Yang     |Australia|Married       |Male  |1971-10-06|2025-10-06 |
|2           |11001      |AW00011001     |Eugene    |Huang    |Australia|Single        |Male  |1976-05-10|2025-10-06 |
|3           |11002      |AW00011002     |Ruben     |Torres   |Australia|Married       |Male  |1971-02-09|2025-10-06 |
|4           |11003      |AW00011003     |Christy   |Zhu      |Australia|Single        |Female|1973-08-14|2025-10-06 |
|5           |11004      |AW00011004     |Elizabeth |Johnson  |Australia|Single        |Female|1979-08-05|2025-10-06 |
+------------+-----------+---------------+------

In [52]:
df_products_raw.show(5, truncate=False)

+-----------+----------+--------------+-------------------------+-----------+----------+--------------+-----------+------+------------+----------+
|product_key|product_id|product_number|product_name             |category_id|category  |subcategory   |maintenance|cost  |product_line|start_date|
+-----------+----------+--------------+-------------------------+-----------+----------+--------------+-----------+------+------------+----------+
|1          |210       |FR-R92B-58    |HL Road Frame - Black- 58|CO_RF      |Components|Road Frames   |Yes        |0.0   |Road        |2003-07-01|
|2          |211       |FR-R92R-58    |HL Road Frame - Red- 58  |CO_RF      |Components|Road Frames   |Yes        |0.0   |Road        |2003-07-01|
|3          |348       |BK-M82B-38    |Mountain-100 Black- 38   |BI_MB      |Bikes     |Mountain Bikes|Yes        |1898.0|Mountain    |2011-07-01|
|4          |349       |BK-M82B-42    |Mountain-100 Black- 42   |BI_MB      |Bikes     |Mountain Bikes|Yes        |189

In [53]:
df_sales_raw.show(5, truncate=False)

+------------+-----------+------------+----------+-------------+----------+------------+--------+-----+
|order_number|product_key|customer_key|order_date|shipping_date|due_date  |sales_amount|quantity|price|
+------------+-----------+------------+----------+-------------+----------+------------+--------+-----+
|SO54496     |282        |5400        |2013-03-16|2013-03-23   |2013-03-28|25.0        |1       |25.0 |
|SO54496     |289        |5400        |2013-03-16|2013-03-23   |2013-03-28|5.0         |1       |5.0  |
|SO54496     |259        |5400        |2013-03-16|2013-03-23   |2013-03-28|2.0         |1       |2.0  |
|SO54497     |174        |9281        |2013-03-16|2013-03-23   |2013-03-28|22.0        |1       |22.0 |
|SO54497     |280        |9281        |2013-03-16|2013-03-23   |2013-03-28|9.0         |1       |9.0  |
+------------+-----------+------------+----------+-------------+----------+------------+--------+-----+
only showing top 5 rows


In [54]:
df_customers_raw.select([
    count(
        when(col(c).isNull(), c)).alias(c)
    for c in df_customers_raw.columns
    ]).show()

+------------+-----------+---------------+----------+---------+-------+--------------+------+---------+-----------+
|customer_key|customer_id|customer_number|first_name|last_name|country|marital_status|gender|birthdate|create_date|
+------------+-----------+---------------+----------+---------+-------+--------------+------+---------+-----------+
|           0|          0|              0|         0|        0|      0|             0|     0|       17|          0|
+------------+-----------+---------------+----------+---------+-------+--------------+------+---------+-----------+



In [55]:
# fillna() replaces NULLs with a default value per-column (dict form)
df_customers_clean = df_customers_raw.fillna({
    "country": "Unknown",
    "gender": "Not Specified",
})
df_customers_clean.show(5)

+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+
|customer_key|customer_id|customer_number|first_name|last_name|  country|marital_status|gender| birthdate|create_date|
+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+
|           1|      11000|     AW00011000|       Jon|     Yang|Australia|       Married|  Male|1971-10-06| 2025-10-06|
|           2|      11001|     AW00011001|    Eugene|    Huang|Australia|        Single|  Male|1976-05-10| 2025-10-06|
|           3|      11002|     AW00011002|     Ruben|   Torres|Australia|       Married|  Male|1971-02-09| 2025-10-06|
|           4|      11003|     AW00011003|   Christy|      Zhu|Australia|        Single|Female|1973-08-14| 2025-10-06|
|           5|      11004|     AW00011004| Elizabeth|  Johnson|Australia|        Single|Female|1979-08-05| 2025-10-06|
+------------+-----------+---------------+------

In [56]:
# drop rows where birthdate is null (can't safely guess a birthdate)
df_customers_clean = df_customers_clean.dropna(subset=["birthdate"])
df_customers_clean.show(5)

+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+
|customer_key|customer_id|customer_number|first_name|last_name|  country|marital_status|gender| birthdate|create_date|
+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+
|           1|      11000|     AW00011000|       Jon|     Yang|Australia|       Married|  Male|1971-10-06| 2025-10-06|
|           2|      11001|     AW00011001|    Eugene|    Huang|Australia|        Single|  Male|1976-05-10| 2025-10-06|
|           3|      11002|     AW00011002|     Ruben|   Torres|Australia|       Married|  Male|1971-02-09| 2025-10-06|
|           4|      11003|     AW00011003|   Christy|      Zhu|Australia|        Single|Female|1973-08-14| 2025-10-06|
|           5|      11004|     AW00011004| Elizabeth|  Johnson|Australia|        Single|Female|1979-08-05| 2025-10-06|
+------------+-----------+---------------+------

In [58]:
# fill nulls
df_products_clean = df_products_raw.fillna({
    "category": "Uncategorized",
    "subcategory": "Uncategorized",
    "maintenance": "Unknown",
    "product_line": "Unknown",
})

In [59]:
# sales: drop rows with a null order_date since it is a required business key
df_sales_clean = df_sales_raw.dropna(subset=["order_date"])

In [60]:
print("customers:", df_customers_clean.count(), "(was", df_customers_raw.count(), ")")

customers: 18467 (was 18484 )


In [61]:
print("products :", df_products_clean.count(),  "(was", df_products_raw.count(),  ")")

products : 295 (was 295 )


In [62]:
print("sales    :", df_sales_clean.count(),      "(was", df_sales_raw.count(),      ")")

sales    : 60379 (was 60398 )


In [65]:
df_customers_selected = df_customers_clean.select(
    col("customer_key"),
    col("customer_id"),
    col("first_name"),
    col("last_name"),
    col("country"),
    col("gender"),
    col("birthdate").alias("date_of_birth"),
)
df_customers_selected.show(5)

+------------+-----------+----------+---------+---------+------+-------------+
|customer_key|customer_id|first_name|last_name|  country|gender|date_of_birth|
+------------+-----------+----------+---------+---------+------+-------------+
|           1|      11000|       Jon|     Yang|Australia|  Male|   1971-10-06|
|           2|      11001|    Eugene|    Huang|Australia|  Male|   1976-05-10|
|           3|      11002|     Ruben|   Torres|Australia|  Male|   1971-02-09|
|           4|      11003|   Christy|      Zhu|Australia|Female|   1973-08-14|
|           5|      11004| Elizabeth|  Johnson|Australia|Female|   1979-08-05|
+------------+-----------+----------+---------+---------+------+-------------+
only showing top 5 rows


In [67]:
df_us_customers = df_customers_selected.filter(col("country") == "United States")
df_us_customers.show(5)

+------------+-----------+----------+---------+-------------+------+-------------+
|customer_key|customer_id|first_name|last_name|      country|gender|date_of_birth|
+------------+-----------+----------+---------+-------------+------+-------------+
|          13|      11012|    Lauren|   Walker|United States|Female|   1979-01-14|
|          14|      11013|       Ian|  Jenkins|United States|  Male|   1979-08-03|
|          15|      11014|    Sydney|  Bennett|United States|Female|   1973-11-06|
|          16|      11015|     Chloe|    Young|United States|Female|   1984-08-26|
|          17|      11016|     Wyatt|     Hill|United States|  Male|   1984-10-25|
+------------+-----------+----------+---------+-------------+------+-------------+
only showing top 5 rows


In [71]:
print("\nUS customers:", df_us_customers.count())


US customers: 7476


In [74]:
df_customers_flagged = df_customers_clean.withColumn(
    "is_married",
    when(col("marital_status") == "Married", "Yes").otherwise("No")
)
df_customers_flagged.show(5, truncate=False)

+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+----------+
|customer_key|customer_id|customer_number|first_name|last_name|country  |marital_status|gender|birthdate |create_date|is_married|
+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+----------+
|1           |11000      |AW00011000     |Jon       |Yang     |Australia|Married       |Male  |1971-10-06|2025-10-06 |Yes       |
|2           |11001      |AW00011001     |Eugene    |Huang    |Australia|Single        |Male  |1976-05-10|2025-10-06 |No        |
|3           |11002      |AW00011002     |Ruben     |Torres   |Australia|Married       |Male  |1971-02-09|2025-10-06 |Yes       |
|4           |11003      |AW00011003     |Christy   |Zhu      |Australia|Single        |Female|1973-08-14|2025-10-06 |No        |
|5           |11004      |AW00011004     |Elizabeth |Johnson  |Australia|Single        |Fe

In [73]:
df_products_tiered = df_products_clean.withColumn(
    "cost_tier",
    when(col("cost") == 0, "Free/Component")
    .when(col("cost") < 500, "Low")
    .when(col("cost") < 2000, "Medium")
    .otherwise("High")
)
df_products_tiered.show(5, truncate=False)

+-----------+----------+--------------+-------------------------+-----------+----------+--------------+-----------+------+------------+----------+--------------+
|product_key|product_id|product_number|product_name             |category_id|category  |subcategory   |maintenance|cost  |product_line|start_date|cost_tier     |
+-----------+----------+--------------+-------------------------+-----------+----------+--------------+-----------+------+------------+----------+--------------+
|1          |210       |FR-R92B-58    |HL Road Frame - Black- 58|CO_RF      |Components|Road Frames   |Yes        |0.0   |Road        |2003-07-01|Free/Component|
|2          |211       |FR-R92R-58    |HL Road Frame - Red- 58  |CO_RF      |Components|Road Frames   |Yes        |0.0   |Road        |2003-07-01|Free/Component|
|3          |348       |BK-M82B-38    |Mountain-100 Black- 38   |BI_MB      |Bikes     |Mountain Bikes|Yes        |1898.0|Mountain    |2011-07-01|Medium        |
|4          |349       |BK-M

In [76]:
df_products_tiered.groupBy("cost_tier").count().show()

+--------------+-----+
|     cost_tier|count|
+--------------+-----+
|          High|    5|
|           Low|  209|
|Free/Component|    2|
|        Medium|   79|
+--------------+-----+



In [77]:
df_sales_casted = df_sales_clean.withColumn(
    "sales_amount", col("sales_amount").cast(DoubleType())
).withColumn(
    "price", col("price").cast(DoubleType())
)

df_sales_casted.show(10, truncate=False)

+------------+-----------+------------+----------+-------------+----------+------------+--------+-----+
|order_number|product_key|customer_key|order_date|shipping_date|due_date  |sales_amount|quantity|price|
+------------+-----------+------------+----------+-------------+----------+------------+--------+-----+
|SO54496     |282        |5400        |2013-03-16|2013-03-23   |2013-03-28|25.0        |1       |25.0 |
|SO54496     |289        |5400        |2013-03-16|2013-03-23   |2013-03-28|5.0         |1       |5.0  |
|SO54496     |259        |5400        |2013-03-16|2013-03-23   |2013-03-28|2.0         |1       |2.0  |
|SO54497     |174        |9281        |2013-03-16|2013-03-23   |2013-03-28|22.0        |1       |22.0 |
|SO54497     |280        |9281        |2013-03-16|2013-03-23   |2013-03-28|9.0         |1       |9.0  |
|SO54498     |174        |4825        |2013-03-16|2013-03-23   |2013-03-28|22.0        |1       |22.0 |
|SO54498     |277        |4825        |2013-03-16|2013-03-23   |

In [79]:
df_sales_dates = df_sales_casted.withColumn(
    "order_year", year(col("order_date"))
).withColumn(
    "order_month", month(col("order_date"))
).withColumn(
    "delivery_days", datediff(col("shipping_date"), col("order_date"))
).withColumn(
    "days_since_order", datediff(current_date(), col("order_date"))
)

df_sales_dates.show(10, truncate=False)

+------------+-----------+------------+----------+-------------+----------+------------+--------+-----+----------+-----------+-------------+----------------+
|order_number|product_key|customer_key|order_date|shipping_date|due_date  |sales_amount|quantity|price|order_year|order_month|delivery_days|days_since_order|
+------------+-----------+------------+----------+-------------+----------+------------+--------+-----+----------+-----------+-------------+----------------+
|SO54496     |282        |5400        |2013-03-16|2013-03-23   |2013-03-28|25.0        |1       |25.0 |2013      |3          |7            |4888            |
|SO54496     |289        |5400        |2013-03-16|2013-03-23   |2013-03-28|5.0         |1       |5.0  |2013      |3          |7            |4888            |
|SO54496     |259        |5400        |2013-03-16|2013-03-23   |2013-03-28|2.0         |1       |2.0  |2013      |3          |7            |4888            |
|SO54497     |174        |9281        |2013-03-16|20

In [80]:
df_sales_dates.select(
    "order_number", "order_date", "shipping_date",
    "order_year", "order_month", "delivery_days"
).show(5, truncate=False)

+------------+----------+-------------+----------+-----------+-------------+
|order_number|order_date|shipping_date|order_year|order_month|delivery_days|
+------------+----------+-------------+----------+-----------+-------------+
|SO54496     |2013-03-16|2013-03-23   |2013      |3          |7            |
|SO54496     |2013-03-16|2013-03-23   |2013      |3          |7            |
|SO54496     |2013-03-16|2013-03-23   |2013      |3          |7            |
|SO54497     |2013-03-16|2013-03-23   |2013      |3          |7            |
|SO54497     |2013-03-16|2013-03-23   |2013      |3          |7            |
+------------+----------+-------------+----------+-----------+-------------+
only showing top 5 rows


In [81]:
df_sales_dates.select(spark_round(avg("delivery_days"), 2).alias("avg_delivery_days")).show()

+-----------------+
|avg_delivery_days|
+-----------------+
|              7.0|
+-----------------+



In [83]:
df_customers_string = df_customers_flagged.withColumn(
    "full_name", initcap(concat_ws(" ", trim(col("first_name")), trim(col("last_name"))))
).withColumn(
    "country_upper", upper(col("country"))
).withColumn(
    "gender_lower", lower(col("gender"))
)
df_customers_string.show(10, truncate=False)

+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+----------+-----------------+-------------+------------+
|customer_key|customer_id|customer_number|first_name|last_name|country  |marital_status|gender|birthdate |create_date|is_married|full_name        |country_upper|gender_lower|
+------------+-----------+---------------+----------+---------+---------+--------------+------+----------+-----------+----------+-----------------+-------------+------------+
|1           |11000      |AW00011000     |Jon       |Yang     |Australia|Married       |Male  |1971-10-06|2025-10-06 |Yes       |Jon Yang         |AUSTRALIA    |male        |
|2           |11001      |AW00011001     |Eugene    |Huang    |Australia|Single        |Male  |1976-05-10|2025-10-06 |No        |Eugene Huang     |AUSTRALIA    |male        |
|3           |11002      |AW00011002     |Ruben     |Torres   |Australia|Married       |Male  |1971-02-09|2025-10-06 |Yes    

In [85]:
df_customers_string.select("first_name", "last_name", "full_name", "country_upper").show(5, truncate=False)

+----------+---------+-----------------+-------------+
|first_name|last_name|full_name        |country_upper|
+----------+---------+-----------------+-------------+
|Jon       |Yang     |Jon Yang         |AUSTRALIA    |
|Eugene    |Huang    |Eugene Huang     |AUSTRALIA    |
|Ruben     |Torres   |Ruben Torres     |AUSTRALIA    |
|Christy   |Zhu      |Christy Zhu      |AUSTRALIA    |
|Elizabeth |Johnson  |Elizabeth Johnson|AUSTRALIA    |
+----------+---------+-----------------+-------------+
only showing top 5 rows


In [88]:
df_sales_full = df_sales_dates.join(
    df_customers_string, "customer_key", "inner"
).join(
    df_products_tiered, "product_key", "inner"
).select(
    df_sales_dates["order_number"],
    df_sales_dates["order_date"],
    df_sales_dates["order_year"],
    df_sales_dates["order_month"],
    df_sales_dates["delivery_days"],
    df_sales_dates["sales_amount"],
    df_sales_dates["quantity"],
    df_sales_dates["price"],
    df_customers_string["customer_key"],
    df_customers_string["full_name"],
    df_customers_string["country"],
    df_customers_string["gender"],
    df_customers_string["is_married"],
    df_products_tiered["product_key"],
    df_products_tiered["product_name"],
    df_products_tiered["category"],
    df_products_tiered["subcategory"],
    df_products_tiered["cost_tier"],
)

df_sales_full.show(10, truncate=False)

+------------+----------+----------+-----------+-------------+------------+--------+-----+------------+----------------+-------------+------+----------+-----------+------------------------------+-----------+---------------+---------+
|order_number|order_date|order_year|order_month|delivery_days|sales_amount|quantity|price|customer_key|full_name       |country      |gender|is_married|product_key|product_name                  |category   |subcategory    |cost_tier|
+------------+----------+----------+-----------+-------------+------------+--------+-----+------------+----------------+-------------+------+----------+-----------+------------------------------+-----------+---------------+---------+
|SO54496     |2013-03-16|2013      |3          |7            |25.0        |1       |25.0 |5400        |Arturo Xu       |Germany      |Male  |No        |282        |LL Mountain Tire              |Accessories|Tires and Tubes|Low      |
|SO54496     |2013-03-16|2013      |3          |7            |5.

In [89]:
df_sales_full.show(5, truncate=False)

+------------+----------+----------+-----------+-------------+------------+--------+-----+------------+-------------+-------+------+----------+-----------+---------------------+-----------+---------------+---------+
|order_number|order_date|order_year|order_month|delivery_days|sales_amount|quantity|price|customer_key|full_name    |country|gender|is_married|product_key|product_name         |category   |subcategory    |cost_tier|
+------------+----------+----------+-----------+-------------+------------+--------+-----+------------+-------------+-------+------+----------+-----------+---------------------+-----------+---------------+---------+
|SO54496     |2013-03-16|2013      |3          |7            |25.0        |1       |25.0 |5400        |Arturo Xu    |Germany|Male  |No        |282        |LL Mountain Tire     |Accessories|Tires and Tubes|Low      |
|SO54496     |2013-03-16|2013      |3          |7            |5.0         |1       |5.0  |5400        |Arturo Xu    |Germany|Male  |No  

In [90]:
print("Joined row count:", df_sales_full.count())

Joined row count: 60300


In [92]:
df_products_never_sold = df_products_tiered.join(
    df_sales_dates, "product_key", "left_anti"
).select("product_key", "product_name", "category")
df_products_never_sold.show(truncate=False)

+-----------+------------------------------+----------+
|product_key|product_name                  |category  |
+-----------+------------------------------+----------+
|1          |HL Road Frame - Black- 58     |Components|
|2          |HL Road Frame - Red- 58       |Components|
|11         |Road-450 Red- 44              |Bikes     |
|12         |Road-450 Red- 48              |Bikes     |
|13         |Road-450 Red- 52              |Bikes     |
|14         |Road-450 Red- 58              |Bikes     |
|15         |Road-450 Red- 60              |Bikes     |
|21         |HL Mountain Frame - Black- 44 |Components|
|22         |HL Mountain Frame - Black- 48 |Components|
|23         |HL Mountain Frame - Silver- 44|Components|
|24         |HL Mountain Frame - Silver- 48|Components|
|25         |ML Road Frame - Red- 44       |Components|
|26         |ML Road Frame - Red- 48       |Components|
|27         |ML Road Frame - Red- 52       |Components|
|28         |ML Road Frame - Red- 58       |Comp

In [93]:
df_products_never_sold.show(10, truncate=False)

+-----------+------------------------------+----------+
|product_key|product_name                  |category  |
+-----------+------------------------------+----------+
|1          |HL Road Frame - Black- 58     |Components|
|2          |HL Road Frame - Red- 58       |Components|
|11         |Road-450 Red- 44              |Bikes     |
|12         |Road-450 Red- 48              |Bikes     |
|13         |Road-450 Red- 52              |Bikes     |
|14         |Road-450 Red- 58              |Bikes     |
|15         |Road-450 Red- 60              |Bikes     |
|21         |HL Mountain Frame - Black- 44 |Components|
|22         |HL Mountain Frame - Black- 48 |Components|
|23         |HL Mountain Frame - Silver- 44|Components|
+-----------+------------------------------+----------+
only showing top 10 rows


In [94]:
print("Count of products never sold:", df_products_never_sold.count())

Count of products never sold: 165


In [96]:
df_revenue_by_country = df_sales_full.groupBy("country").agg(
    count("order_number").alias("total_orders"),
    spark_sum("sales_amount").alias("total_revenue"),
    spark_round(avg("sales_amount"), 2).alias("avg_order_value"),
    count_distinct("customer_key").alias("unique_customers"),
).orderBy(col("total_revenue").desc())

df_revenue_by_country.show(truncate=False)

+--------------+------------+-------------+---------------+----------------+
|country       |total_orders|total_revenue|avg_order_value|unique_customers|
+--------------+------------+-------------+---------------+----------------+
|United States |20444       |9141230.0    |447.14         |7475            |
|Australia     |13327       |9039967.0    |678.32         |3588            |
|United Kingdom|6892        |3383209.0    |490.89         |1912            |
|Germany       |5614        |2891876.0    |515.12         |1777            |
|France        |5533        |2634419.0    |476.13         |1805            |
|Canada        |7619        |1977733.0    |259.58         |1571            |
|n/a           |871         |226820.0     |260.41         |337             |
+--------------+------------+-------------+---------------+----------------+



In [97]:
df_revenue_by_category = df_sales_full.groupBy("category").agg(
    spark_sum("quantity").alias("total_units_sold"),
    spark_sum("sales_amount").alias("total_revenue"),
    spark_round(avg("sales_amount"), 2).alias("avg_sale_amount"),
).orderBy(col("total_revenue").desc())

df_revenue_by_category.show(truncate=False)

+-----------+----------------+-------------+---------------+
|category   |total_units_sold|total_revenue|avg_sale_amount|
+-----------+----------------+-------------+---------------+
|Bikes      |15178           |2.8256931E7  |1861.7         |
|Accessories|36056           |699127.0     |19.4           |
|Clothing   |9091            |339196.0     |37.33          |
+-----------+----------------+-------------+---------------+



In [99]:
df_customer_categories = df_sales_full.groupBy("customer_key", "full_name").agg(
    collect_set("category").alias("categories_purchased"),
    count("order_number").alias("total_orders")
).orderBy(col("total_orders").desc())

df_customer_categories.show(10, truncate=False)

+------------+----------------+-----------------------+------------+
|customer_key|full_name       |categories_purchased   |total_orders|
+------------+----------------+-----------------------+------------+
|186         |Ashley Henderson|[Accessories, Clothing]|68          |
|301         |Fernando Barnes |[Accessories, Clothing]|67          |
|278         |Charles Jackson |[Accessories, Clothing]|65          |
|263         |Jennifer Simmons|[Accessories, Clothing]|63          |
|288         |Henry Garcia    |[Accessories, Clothing]|62          |
|177         |Mason Roberts   |[Accessories, Clothing]|60          |
|92          |Dalton Perez    |[Accessories, Clothing]|59          |
|332         |Samantha Jenkins|[Accessories, Clothing]|58          |
|567         |April Shan      |[Accessories, Clothing]|58          |
|277         |Nancy Chapman   |[Accessories, Clothing]|57          |
+------------+----------------+-----------------------+------------+
only showing top 10 rows


In [101]:
customer_spend = df_sales_full.groupBy("country", "customer_key", "full_name").agg(
    spark_sum("sales_amount").alias("total_spent")
)

customer_spend.show(10, truncate=False)

+-------------+------------+----------------+-----------+
|country      |customer_key|full_name       |total_spent|
+-------------+------------+----------------+-----------+
|Canada       |3354        |Erin Reed       |129.0      |
|United States|17829       |Andrea Murphy   |4329.0     |
|Australia    |98          |Edwin Nara      |8099.0     |
|United States|2182        |Brian Morgan    |186.0      |
|Australia    |15774       |Meghan Martin   |70.0       |
|United States|12895       |Bryan Ramirez   |70.0       |
|Canada       |186         |Ashley Henderson|1616.0     |
|United States|6489        |Jason Adams     |5.0        |
|France       |5296        |Roger Xu        |138.0      |
|Germany      |16002       |Carla Kapoor    |30.0       |
+-------------+------------+----------------+-----------+
only showing top 10 rows


In [102]:
country_window = Window.partitionBy("country").orderBy(col("total_spent").desc())

df_customer_ranked = customer_spend.withColumn(
    "rank_in_country", rank().over(country_window)
).withColumn(
    "dense_rank_in_country", dense_rank().over(country_window)
).withColumn(
    "row_num", row_number().over(country_window)
)

df_customer_ranked.show(10, truncate=False)

+---------+------------+-------------+-----------+---------------+---------------------+-------+
|country  |customer_key|full_name    |total_spent|rank_in_country|dense_rank_in_country|row_num|
+---------+------------+-------------+-----------+---------------+---------------------+-------+
|Australia|768         |Meagan Madan |8319.0     |1              |1                    |1      |
|Australia|113         |Crystal Wang |8295.0     |2              |2                    |2      |
|Australia|767         |Candace Raman|8280.0     |3              |3                    |3      |
|Australia|1339        |Monica Vance |8265.0     |4              |4                    |4      |
|Australia|102         |Abby Sai     |8262.0     |5              |5                    |5      |
|Australia|901         |Byron Carlson|8257.0     |6              |6                    |6      |
|Australia|1004        |Audrey Munoz |8256.0     |7              |7                    |7      |
|Australia|452         |Ruben 

In [103]:
df_customer_ranked.filter(col("rank_in_country") == 1) \
    .orderBy(col("total_spent").desc()).show(10, truncate=False)

+--------------+------------+-----------------+-----------+---------------+---------------------+-------+
|country       |customer_key|full_name        |total_spent|rank_in_country|dense_rank_in_country|row_num|
+--------------+------------+-----------------+-----------+---------------+---------------------+-------+
|France        |1133        |Kaitlyn Henderson|13294.0    |1              |1                    |1      |
|France        |1302        |Nichole Nara     |13294.0    |1              |1                    |2      |
|Germany       |246         |Ricky Vazquez    |10580.0    |1              |1                    |1      |
|United Kingdom|2601        |Marie Sanz       |8460.0     |1              |1                    |1      |
|Australia     |768         |Meagan Madan     |8319.0     |1              |1                    |1      |
|United States |260         |Victoria Stewart |6770.0     |1              |1                    |1      |
|Canada        |11996       |Isabella Bryant  

In [104]:
monthly_revenue = df_sales_full.groupBy("order_year", "order_month").agg(
    spark_sum("sales_amount").alias("monthly_revenue")
).orderBy("order_year", "order_month")

monthly_revenue.show(10, truncate=False)

+----------+-----------+---------------+
|order_year|order_month|monthly_revenue|
+----------+-----------+---------------+
|2010      |12         |43419.0        |
|2011      |1          |469795.0       |
|2011      |2          |466307.0       |
|2011      |3          |485165.0       |
|2011      |4          |502042.0       |
|2011      |5          |561647.0       |
|2011      |6          |737793.0       |
|2011      |7          |596710.0       |
|2011      |8          |610938.0       |
|2011      |9          |603047.0       |
+----------+-----------+---------------+
only showing top 10 rows


In [108]:
time_window = Window.orderBy("order_year", "order_month") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

monthly_lag_window = Window.orderBy("order_year", "order_month")

In [109]:
df_monthly_trend = monthly_revenue.withColumn(
    "running_total_revenue", spark_sum("monthly_revenue").over(time_window)
).withColumn(
    "previous_month_revenue", lag("monthly_revenue", 1).over(monthly_lag_window)
).withColumn(
    "next_month_revenue", lead("monthly_revenue", 1).over(monthly_lag_window)
)

In [111]:
df_monthly_trend.show(12, truncate=False)

+----------+-----------+---------------+---------------------+----------------------+------------------+
|order_year|order_month|monthly_revenue|running_total_revenue|previous_month_revenue|next_month_revenue|
+----------+-----------+---------------+---------------------+----------------------+------------------+
|2010      |12         |43419.0        |43419.0              |NULL                  |469795.0          |
|2011      |1          |469795.0       |513214.0             |43419.0               |466307.0          |
|2011      |2          |466307.0       |979521.0             |469795.0              |485165.0          |
|2011      |3          |485165.0       |1464686.0            |466307.0              |502042.0          |
|2011      |4          |502042.0       |1966728.0            |485165.0              |561647.0          |
|2011      |5          |561647.0       |2528375.0            |502042.0              |737793.0          |
|2011      |6          |737793.0       |3266168.0      

In [113]:
df_pivot_country_category = df_sales_full.groupBy("country") \
    .pivot("category") \
    .agg(spark_round(spark_sum("sales_amount"), 2))

df_pivot_country_category.show(truncate=False)

+--------------+-----------+---------+--------+
|country       |Accessories|Bikes    |Clothing|
+--------------+-----------+---------+--------+
|Australia     |138460.0   |8831331.0|70176.0 |
|United Kingdom|76385.0    |3274678.0|32146.0 |
|United States |248175.0   |8766716.0|126339.0|
|France        |62941.0    |2544710.0|26768.0 |
|n/a           |7749.0     |211866.0 |7205.0  |
|Germany       |62029.0    |2806434.0|23413.0 |
|Canada        |103388.0   |1821196.0|53149.0 |
+--------------+-----------+---------+--------+



In [122]:
df_sales_full.createOrReplaceTempView("vw_sales_full")
df_customers_string.createOrReplaceTempView("vw_customers")
df_products_tiered.createOrReplaceTempView("vw_products")
customer_spend.createOrReplaceTempView("vw_customer_spend")

In [124]:
# Total revenue and orders per year
spark.sql("""
    SELECT order_year,
           COUNT(order_number)            AS total_orders,
           ROUND(SUM(sales_amount), 2)    AS total_revenue
    FROM vw_sales_full
    GROUP BY order_year
    ORDER BY order_year
""").show()

+----------+------------+-------------+
|order_year|total_orders|total_revenue|
+----------+------------+-------------+
|      2010|          14|      43419.0|
|      2011|        2210|    7054204.0|
|      2012|        3391|    5833004.0|
|      2013|       52715|  1.6318985E7|
|      2014|        1970|      45642.0|
+----------+------------+-------------+



In [125]:
# Top 5 best-selling products by revenue
spark.sql("""
    SELECT product_name,
           category,
           SUM(quantity)                 AS units_sold,
           ROUND(SUM(sales_amount), 2)   AS total_revenue
    FROM vw_sales_full
    GROUP BY product_name, category
    ORDER BY total_revenue DESC
    LIMIT 5
""").show(truncate=False)

+-----------------------+--------+----------+-------------+
|product_name           |category|units_sold|total_revenue|
+-----------------------+--------+----------+-------------+
|Mountain-200 Black- 46 |Bikes   |619       |1371159.0    |
|Mountain-200 Black- 42 |Bikes   |610       |1354686.0    |
|Mountain-200 Silver- 38|Bikes   |594       |1334754.0    |
|Mountain-200 Silver- 46|Bikes   |578       |1296389.0    |
|Mountain-200 Black- 38 |Bikes   |579       |1287969.0    |
+-----------------------+--------+----------+-------------+



In [126]:
# Customers who spent above the overall average
spark.sql("""
    SELECT full_name, country, total_spent
    FROM vw_customer_spend
    WHERE total_spent > (SELECT AVG(total_spent) FROM vw_customer_spend)
    ORDER BY total_spent DESC
    LIMIT 10
""").show(truncate=False)

+-----------------+-------+-----------+
|full_name        |country|total_spent|
+-----------------+-------+-----------+
|Kaitlyn Henderson|France |13294.0    |
|Nichole Nara     |France |13294.0    |
|Margaret He      |France |13268.0    |
|Randall Dominguez|France |13265.0    |
|Adriana Gonzalez |France |13242.0    |
|Rosa Hu          |France |13215.0    |
|Brandi Gill      |France |13195.0    |
|Brad She         |France |13172.0    |
|Francisco Sara   |France |13164.0    |
|Maurice Shan     |France |12914.0    |
+-----------------+-------+-----------+



In [129]:
# Ranking customers within each country
spark.sql("""
    SELECT country, full_name, total_spent,
           RANK() OVER (PARTITION BY country ORDER BY total_spent DESC) AS country_rank
    FROM vw_customer_spend
    QUALIFY country_rank <= 3
    ORDER BY country, country_rank
""").show(30, truncate=False)

+--------------+------------------+-----------+------------+
|country       |full_name         |total_spent|country_rank|
+--------------+------------------+-----------+------------+
|Australia     |Meagan Madan      |8319.0     |1           |
|Australia     |Crystal Wang      |8295.0     |2           |
|Australia     |Candace Raman     |8280.0     |3           |
|Canada        |Isabella Bryant   |6060.0     |1           |
|Canada        |Chloe Miller      |6056.0     |2           |
|Canada        |Hailey Stewart    |6054.0     |3           |
|France        |Kaitlyn Henderson |13294.0    |1           |
|France        |Nichole Nara      |13294.0    |1           |
|France        |Margaret He       |13268.0    |3           |
|Germany       |Ricky Vazquez     |10580.0    |1           |
|Germany       |Latasha Rubio     |10575.0    |2           |
|Germany       |Clarence Anand    |10566.0    |3           |
|United Kingdom|Marie Sanz        |8460.0     |1           |
|United Kingdom|Emmanuel

In [131]:
# Category revenue share of total revenue
spark.sql("""
    WITH category_totals AS (
        SELECT category, SUM(sales_amount) AS category_revenue
        FROM vw_sales_full
        GROUP BY category
    ),
    grand_total AS (
        SELECT SUM(sales_amount) AS overall_revenue FROM vw_sales_full
    )
    SELECT c.category,
           ROUND(c.category_revenue, 2)                                   AS category_revenue,
           ROUND(c.category_revenue * 100.0 / g.overall_revenue, 2)       AS pct_of_total
    FROM category_totals c
    CROSS JOIN grand_total g
    ORDER BY category_revenue DESC
""").show(truncate=False)

+-----------+----------------+------------+
|category   |category_revenue|pct_of_total|
+-----------+----------------+------------+
|Bikes      |2.8256931E7     |96.46       |
|Accessories|699127.0        |2.39        |
|Clothing   |339196.0        |1.16        |
+-----------+----------------+------------+



In [135]:
# save the output as parquet file 
output_path = "output/gold_sales_enriched_parquet"
df_sales_full.write.mode("overwrite").partitionBy("order_year").parquet(output_path)
print(f"\nEnriched sales table written to: {output_path} (partitioned by order_year)")

Py4JJavaError: An error occurred while calling o1204.parquet.
: org.apache.spark.SparkException: [INTERNAL_ERROR] Executed command failed. You hit a bug in Spark or the Spark plugins you use. Please, report this bug to the corresponding communities or vendors, and provide the full stack trace. SQLSTATE: XX000
	at org.apache.spark.SparkException$.internalError(SparkException.scala:119)
	at org.apache.spark.sql.execution.QueryExecution$.toInternalError(QueryExecution.scala:861)
	at org.apache.spark.sql.execution.QueryExecution$.withInternalError(QueryExecution.scala:874)
	at org.apache.spark.sql.execution.QueryExecution$.runCommand(QueryExecution.scala:939)
	at org.apache.spark.sql.execution.QueryExecution.org$apache$spark$sql$execution$QueryExecution$$eagerlyExecute$1(QueryExecution.scala:247)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$eagerlyExecuteCommands$1.applyOrElse(QueryExecution.scala:261)
	at org.apache.spark.sql.execution.QueryExecution$$anonfun$eagerlyE

In [133]:
# stop spark
spark.stop()